# QuantumCircuit.jl Phase 2 Tunable Sweep Walkthrough

**Audience**
- Julia users who already know the Phase 1 workflow and want to use tunable devices and sweep analysis.

**Prerequisites**
- Basic Julia syntax.
- Familiarity with `CompositeSystem`, `spectrum`, and `transition_frequencies`.
- Basic intuition for reduced flux and SQUID tunability.

**What you will learn**
- How to define a `TunableTransmon` and build a flux sweep.
- How to summarize a sweep with `transition_curve`, `anharmonicity_curve`, `minimum_gap`, and `sweep_summary`.
- How to run a coupling sweep with the same `SweepSpec` interface.
- How to inspect the updated model stored inside each `SpectrumResult`.


## Outline

1. Activate the local package environment.
2. Create a tunable subsystem and run a flux sweep.
3. Read the sweep with the current Analysis helpers.
4. Run a coupling sweep in a coupled system.
5. Inspect the applied sweep parameters directly from the results.
6. Review pitfalls and try a small exercise.


In [2]:
using Pkg

function find_repo_root(start::AbstractString)
    candidates = [
        normpath(start),
        normpath(joinpath(start, "..")),
        normpath(joinpath(start, "..", "..")),
    ]

    for candidate in unique(candidates)
        project_toml = joinpath(candidate, "Project.toml")
        if isfile(project_toml)
            content = read(project_toml, String)
            occursin("QuantumCircuit", content) && return candidate
        end
    end

    error("Could not find the QuantumCircuit.jl project root. Start Jupyter from the repository or open the notebook from inside it.")
end

project_root = find_repo_root(pwd())
Pkg.activate(project_root)
Pkg.instantiate()

using QuantumCircuit


  Activating project at `~/Research/20_Projects/QuantumCircuit.jl`


## Step 1 - Define a tunable subsystem and flux sweep

Phase 2 keeps the same `CompositeSystem` entry point but adds tunable subsystem types.

**Unit convention**
- `EJmax`, `EC`, resonator `ω`, and coupling `g` should use the same frequency unit.
- This walkthrough uses `GHz`, so reported energies and transition frequencies are also in `GHz`.
- `flux` is dimensionless reduced flux `Phi/Phi0`.
- `asymmetry` is a dimensionless SQUID asymmetry parameter.


In [3]:
tq = TunableTransmon(:tq; EJmax = 20.0, EC = 0.25, flux = 0.0, asymmetry = 0.05, ncut = 6)
tunable_sys = CompositeSystem(tq)

flux_values = [0.0, 0.10, 0.20, 0.30]
flux_sweep = SweepSpec(:tq, :flux, flux_values; levels = 4)
flux_result = simulate_sweep(tunable_sys, flux_sweep)

(
    subsystem_names = subsystem_names(tunable_sys),
    sweep_values = flux_result.values,
    first_spectrum = flux_result.spectra[1].energies,
)


(subsystem_names = [:tq], sweep_values = [0.0, 0.1, 0.2, 0.3], first_spectrum = [0.0, 6.074555320336759, 11.899110640673522, 17.473665961010273])

## Step 2 - Summarize the sweep with Analysis helpers

The current Analysis layer is result-driven. That means the helpers read `SpectrumResult` and `SweepResult` directly, instead of rebuilding Hamiltonians or rerunning solvers.


In [4]:
ω01_curve = transition_curve(flux_result)
alpha_curve = anharmonicity_curve(flux_result)
gap = minimum_gap(flux_result)
summary = sweep_summary(flux_result)

(
    label = ω01_curve.label,
    flux_values = summary.values,
    ω01 = ω01_curve.data,
    α = alpha_curve.data,
    minimum_gap = gap,
)


(label = :transition_01, flux_values = [0.0, 0.1, 0.2, 0.3], ω01 = [6.074555320336759, 5.9182477743678135, 5.440520651422999, 4.604587535975373], α = [-0.24999999999999645, -0.24999999999999822, -0.24999999999999822, -0.24999999999999822], minimum_gap = (gap = 4.604587535975373, sweep_value = 0.3, index = 4, level_pair = (1, 2)))

## Step 3 - Sweep a coupling parameter in a coupled system

The same `SweepSpec` entry point also works for couplings. Here the sweep target is identified by the two subsystem names attached to the coupling.


In [5]:
q1 = Transmon(:q1; EJ = 20.0, EC = 0.25, ncut = 5)
r1 = Resonator(:r1; ω = 6.8, dim = 4)

coupled_sys = CompositeSystem(
    q1,
    r1,
    CapacitiveCoupling(:q1, :r1; g = 0.02),
)

g_values = [0.02, 0.08, 0.14]
g_sweep = SweepSpec(:q1, :r1, :g, g_values; levels = 5)
g_result = simulate_sweep(coupled_sys, g_sweep)
g_curve = transition_curve(g_result)
g_summary = sweep_summary(g_result)

(
    coupling_values = g_summary.values,
    ω01 = g_curve.data,
    weakest_coupling_spectrum = g_result.spectra[1].energies,
)


(coupling_values = [0.02, 0.08, 0.14], ω01 = [6.074004352846019, 6.065837899489844, 6.0484750163850265], weakest_coupling_spectrum = [0.0, 6.074004352846019, 6.800550967490739, 11.898290795844863, 12.874273352821513])

## Step 4 - Inspect the updated model stored in each result

Each `SpectrumResult` keeps the model that produced it. This makes it easy to verify which parameter value was actually applied at a given sweep point.


In [6]:
(
    swept_flux = flux_result.spectra[3].model.system.subsystems[1].flux,
    swept_coupling = g_result.spectra[3].model.system.couplings[1].g,
    flux_curve_label = ω01_curve.label,
    coupling_curve_label = g_curve.label,
)


(swept_flux = 0.2, swept_coupling = 0.14, flux_curve_label = :transition_01, coupling_curve_label = :transition_01)

## Pitfalls and Current Limits

**Common pitfall**
- Sweeping flux all the way to `0.5` with zero asymmetry can drive the effective Josephson energy too low for the current Duffing approximation.

**Current model assumption**
- The tunable path still uses a Duffing-style approximation with a SQUID effective `EJ`.

**Practical reading**
- Treat this notebook as a solid Phase 2 workflow, not yet as a full circuit-quantization or dynamics workflow.


In [7]:
problematic = TunableTransmon(:bad; EJmax = 20.0, EC = 0.25, flux = 0.5, asymmetry = 0.0, ncut = 6)

try
    hamiltonian(CompositeSystem(problematic))
catch err
    sprint(showerror, err)
end


"ArgumentError: Duffing approximation for bad requires a positive local frequency; choose parameters with larger effective EJ."

## Exercise

Try one of the following:

1. Change `flux_values` to a finer grid and inspect how the `minimum_gap` result changes.
2. Change `asymmetry` from `0.05` to `0.2` and compare `transition_curve`.
3. Replace the flux sweep with a sweep over the coupling `g` in a larger composite system.

Before running the next cell, predict whether larger asymmetry will make the half-flux region less singular.


In [8]:
exercise_tq = TunableTransmon(:tq; EJmax = 20.0, EC = 0.25, flux = 0.0, asymmetry = 0.2, ncut = 6)
exercise_sys = CompositeSystem(exercise_tq)
exercise_sweep = SweepSpec(:tq, :flux, [0.0, 0.15, 0.30, 0.45]; levels = 4)
exercise_result = simulate_sweep(exercise_sys, exercise_sweep)

(
    asymmetry = exercise_tq.asymmetry,
    ω01 = transition_curve(exercise_result).data,
    minimum_gap = minimum_gap(exercise_result),
)


(asymmetry = 0.2, ω01 = [6.074555320336759, 5.735385433041085, 4.68821405117471, 2.9247624794395577], minimum_gap = (gap = 2.9247624794395577, sweep_value = 0.45, index = 4, level_pair = (1, 2)))